In [30]:
import pandas as pd

ns_dataset = pd.read_csv(r'.\final_dns_pipeline_multi_ns_cleaned_with_revenue_sorted.csv')

#carregar o dataset de PFX to AS com colunas distribuidas como o exemplo 1.0.0.0	24	13335

pfx_to_as_dataset = pd.read_csv(r'.\routeviews-rv2-20250930-2200.pfx2as', sep='\t', header=None, names=['prefix', 'prefix_length', 'as_number'])

#carregar o dataset de AS to Org com colunas as_number|changed|aut_name|org_id|opaque_id|source

as_to_org_dataset = pd.read_csv(r'.\20251001.as2org.txt', sep='|', header=1, names=['as_number', 'changed', 'aut_name', 'org_id', 'opaque_id', 'source'])

#carregar o dataset de AS-ORG to Info com colunas org_id|changed|org_name|country|source

as_org_to_info_dataset = pd.read_csv(r'.\20251001.as-org2info.txt', sep='|', header=1, names=['org_id', 'changed', 'org_name', 'country', 'source'])


display(ns_dataset.head(10))
print(f"Number of unique domains with revenue Greater than 0: {ns_dataset[ns_dataset['revenue'] > 0]['domain'].nunique()}")
display(pfx_to_as_dataset.head(10))
display(as_to_org_dataset.head(10))
display(as_org_to_info_dataset.head(10))

,domain,NS,NS-A,company_name,revenue,country_name,sector,rank
0,walmart.com,a5-65.akam.net,95.100.168.65,Walmart,6.931530e+11,United States,Consumer Staples,1.0
1,walmart.com,pdnswm4.ultradns.org,156.154.67.4,Walmart,6.931530e+11,United States,Consumer Staples,1.0
2,walmart.com,a8-66.akam.net,2.16.40.66,Walmart,6.931530e+11,United States,Consumer Staples,1.0
3,amazon.com,ns2.amzndns.net,156.154.69.10,Amazon,4.447496e+11,China,Energy,4.0
4,unitedhealthgroup.com,epla1.corpnamesvcs.com,149.111.128.204,UnitedHealth,4.228180e+11,United States,Health Care,5.0
5,unitedhealthgroup.com,epla2.corpnamesvcs.com,168.183.36.43,UnitedHealth,4.228180e+11,United States,Health Care,5.0
6,apple.com,a.ns.apple.com,17.253.200.1,Apple,4.086250e+11,United States,Information Technology,6.0
7,apple.com,d.ns.apple.com,204.26.57.1,Apple,4.086250e+11,United States,Information Technology,6.0
8,apple.com,c.ns.apple.com,204.19.119.1,Apple,4.086250e+11,United States,Information Technology,6.0
9,berkshirehathaway.com,ns9.worldnic.com,162.159.26.110,Berkshire Hathaway,3.971460e+11,United States,Financials,7.0


Number of unique domains with revenue Greater than 0: 10374


,prefix,prefix_length,as_number
0,1.0.0.0,24,13335
1,1.0.4.0,24,38803
2,1.0.5.0,24,38803
3,1.0.6.0,24,38803
4,1.0.7.0,24,38803
5,1.0.16.0,24,2519
6,1.0.64.0,18,18144
7,1.0.128.0,17,23969
8,1.0.128.0,18,23969
9,1.0.128.0,19,23969


,as_number,changed,aut_name,org_id,opaque_id,source
0,2,20231108.0,UDEL-DCN,UNIVER-19-Z-ARIN,279fb28df10add3bd7028865951995a6_ARIN,ARIN
1,3,20100927.0,MIT-GATEWAYS,MIT-2-ARIN,d98c567cda2db06e693f2b574eafe848_ARIN,ARIN
2,4,20230929.0,ISI-AS,USC-32-Z-ARIN,8f5d315929a560376b0b58b40a1932fa_ARIN,ARIN
3,5,20200723.0,SYMBOLICS,WGL-117-ARIN,481404355c401f2604c57a0fda4ee68f_ARIN,ARIN
4,6,20210121.0,BULL-HN,ATOS-Z-ARIN,61457436058bce0a6be1a923532a3255_ARIN,ARIN
5,7,NaN,DSTL,ORG-TDSA4-RIPE,NaN,RIPE
6,8,19971110.0,RICE-AS,RICEUN-ARIN,5f676a1dae02fc7cb708558c3ff1d122_ARIN,ARIN
7,9,20120402.0,CMU-ROUTER,CARNEG-Z-ARIN,859ff8395a142b506a4aa4425d450e1d_ARIN,ARIN
8,10,20000418.0,CSNET-EXT-AS,CCICC-ARIN,3fa2e5aa48f205a7696ea6fbcd437cff_ARIN,ARIN
9,11,20231221.0,HARVARD,HARVAR-Z-ARIN,2c8438ff5e93d4a88cf7217faf2d6b5c_ARIN,ARIN


,org_id,changed,org_name,country,source
0,1800FL-ARIN,20250812.0,"1-800-Flowers.com, Inc.",US,ARIN
1,1800H-ARIN,20160520.0,"1-800-HOSTING, Inc.",US,ARIN
2,1881CS-ARIN,20121010.0,1881CS/XPC,US,ARIN
3,1FBU-ARIN,20180521.0,1st Financial Bank USA,US,ARIN
4,1GPUC-ARIN,20160531.0,"1798 Global Partners (USA), Corp.",US,ARIN
5,1STP-ARIN,20241125.0,"FIRST STEP INTERNET, LLC",US,ARIN
6,1UWEB-ARIN,20250428.0,"1U Web, INC.",US,ARIN
7,280INC-ARIN,20171026.0,"280, Inc.",US,ARIN
8,2CC-6-ARIN,20240130.0,281 Communications Corporation Inc.,US,ARIN
9,2CHEC-ARIN,20191217.0,2checkout.com,US,ARIN


In [53]:
import re
import pandas as pd

ipv4_re = re.compile(r'(?<!\d)(\d{1,3}(?:\.\d{1,3}){3})(?!\d)')

def normalize_prefix(cell):
    if pd.isna(cell):
        return None
    txt = str(cell).strip()
    # remove máscara se houver
    txt = txt.split("/", 1)[0]
    # pega o primeiro IPv4 que aparecer
    m = ipv4_re.search(txt)
    if not m:
        return None
    ip = m.group(1)
    parts = ip.split(".")
    # checa 0–255
    try:
        nums = list(map(int, parts))
    except ValueError:
        return None
    if any(n < 0 or n > 255 for n in nums):
        return None
    return f"{nums[0]}.{nums[1]}.{nums[2]}.0"

ns_dataset["prefix_norm"] = ns_dataset["NS-A"].apply(normalize_prefix)

# 2) Realizar o merge usando a nova coluna
merged_ns_pfx = pd.merge(
    ns_dataset,
    pfx_to_as_dataset,
    left_on='prefix_norm',
    right_on='prefix',
    how='left'
)

#ns casos em que "aas_number" possui multiplos valores no seguinte formato "399154_399159_399163_399160_399165, usar apenas o primeiro valor antes do "_"
def extract_first_as_number(as_number):
    if pd.isna(as_number):
        return as_number
    return str(as_number).split('_')[0]
merged_ns_pfx['as_number'] = merged_ns_pfx['as_number'].apply(extract_first_as_number)

print(f"Number of unique domains with revenue Greater than 0: {merged_ns_pfx[merged_ns_pfx['revenue'] > 0]['domain'].nunique()}")
display(merged_ns_pfx.head(10))

##list lines without as_number matched

lines_without_as_number = merged_ns_pfx[merged_ns_pfx['as_number'].isna()]
print(f"Number of lines without as_number matched: {len(lines_without_as_number)}")

#save in csv merged_ns_pfx with revenue greater than 0

merged_ns_pfx[merged_ns_pfx['revenue'] > 0].to_csv(r'.\merged_ns_pfx_with_revenue_greater_than_0.csv', index=False)


#liste os dominios que não tiveram NENHUMA correspondência de as_number, faça isso excluindo domínios que possuem ao menos algum as_number correspondente em outra linha
domains_without_any_as_number = merged_ns_pfx[merged_ns_pfx['as_number'].isna()]['domain'].unique()
domains_with_as_number = merged_ns_pfx[merged_ns_pfx['as_number'].notna()]['domain'].unique()

final_domains_without_any_as_number = set(domains_without_any_as_number) - set(domains_with_as_number)
print(f"Number of unique domains without any as_number matched: {len(final_domains_without_any_as_number)}")

# save as lines_without_as_number_not_matched.csv the ones from final_domains_without_any_as_number with revenue greater than 0, choosing only the first occurrence of each domain

lines_without_as_number_not_matched = merged_ns_pfx[(merged_ns_pfx['domain'].isin(final_domains_without_any_as_number)) & (merged_ns_pfx['revenue'] > 0)]
lines_without_as_number_not_matched = lines_without_as_number_not_matched.drop_duplicates(subset=['domain'])
lines_without_as_number_not_matched.to_csv(r'.\lines_without_as_number_not_matched.csv', index=False)

Number of unique domains with revenue Greater than 0: 10374


,domain,NS,NS-A,company_name,revenue,country_name,sector,rank,prefix_norm,prefix,prefix_length,as_number
0,walmart.com,a5-65.akam.net,95.100.168.65,Walmart,6.931530e+11,United States,Consumer Staples,1.0,95.100.168.0,95.100.168.0,24.0,21342
1,walmart.com,pdnswm4.ultradns.org,156.154.67.4,Walmart,6.931530e+11,United States,Consumer Staples,1.0,156.154.67.0,156.154.67.0,24.0,12008
2,walmart.com,a8-66.akam.net,2.16.40.66,Walmart,6.931530e+11,United States,Consumer Staples,1.0,2.16.40.0,2.16.40.0,24.0,21342
3,amazon.com,ns2.amzndns.net,156.154.69.10,Amazon,4.447496e+11,China,Energy,4.0,156.154.69.0,156.154.69.0,24.0,12008
4,unitedhealthgroup.com,epla1.corpnamesvcs.com,149.111.128.204,UnitedHealth,4.228180e+11,United States,Health Care,5.0,149.111.128.0,149.111.128.0,19.0,10879
5,unitedhealthgroup.com,epla2.corpnamesvcs.com,168.183.36.43,UnitedHealth,4.228180e+11,United States,Health Care,5.0,168.183.36.0,168.183.36.0,22.0,10879
6,apple.com,a.ns.apple.com,17.253.200.1,Apple,4.086250e+11,United States,Information Technology,6.0,17.253.200.0,17.253.200.0,24.0,714
7,apple.com,d.ns.apple.com,204.26.57.1,Apple,4.086250e+11,United States,Information Technology,6.0,204.26.57.0,204.26.57.0,24.0,42
8,apple.com,c.ns.apple.com,204.19.119.1,Apple,4.086250e+11,United States,Information Technology,6.0,204.19.119.0,204.19.119.0,24.0,42
9,berkshirehathaway.com,ns9.worldnic.com,162.159.26.110,Berkshire Hathaway,3.971460e+11,United States,Financials,7.0,162.159.26.0,162.159.26.0,24.0,13335


Number of lines without as_number matched: 8651
Number of unique domains without any as_number matched: 2615


In [42]:
merged_ns_pfx["as_number"] = merged_ns_pfx["as_number"].astype(str)
as_to_org_dataset["as_number"] = as_to_org_dataset["as_number"].astype(str)

merged_ns_as_org = pd.merge(
    merged_ns_pfx,
    as_to_org_dataset,
    on="as_number",
    how="left"
)
print(f"Number of unique domains with revenue Greater than 0: {merged_ns_as_org[merged_ns_as_org['revenue'] > 0]['domain'].nunique()}")
display(merged_ns_as_org.head(10))


Number of unique domains with revenue Greater than 0: 10374


,domain,NS,NS-A,company_name,revenue,country_name,sector,rank,prefix,prefix_length,as_number,changed,aut_name,org_id,opaque_id,source
0,walmart.com,a5-65.akam.net,95.100.168.65,Walmart,6.931530e+11,United States,Consumer Staples,1.0,95.100.168.0,24.0,21342,NaN,AKAMAI-ASN2,ORG-AT1-RIPE,NaN,RIPE
1,walmart.com,pdnswm4.ultradns.org,156.154.67.4,Walmart,6.931530e+11,United States,Consumer Staples,1.0,156.154.67.0,24.0,12008,20221213.0,SECURITYSERVICES,SSL-1134-ARIN,d46edd86fd49dba22360500c6b8657bf_ARIN,ARIN
2,walmart.com,a8-66.akam.net,2.16.40.66,Walmart,6.931530e+11,United States,Consumer Staples,1.0,2.16.40.0,24.0,21342,NaN,AKAMAI-ASN2,ORG-AT1-RIPE,NaN,RIPE
3,amazon.com,ns2.amzndns.net,156.154.69.10,Amazon,4.447496e+11,China,Energy,4.0,156.154.69.0,24.0,12008,20221213.0,SECURITYSERVICES,SSL-1134-ARIN,d46edd86fd49dba22360500c6b8657bf_ARIN,ARIN
4,unitedhealthgroup.com,epla1.corpnamesvcs.com,149.111.128.204,UnitedHealth,4.228180e+11,United States,Health Care,5.0,149.111.128.0,19.0,10879,20111213.0,UHC,UNITED-215-ARIN,d39768e24c8472eb3b02486a17098b02_ARIN,ARIN
5,unitedhealthgroup.com,epla2.corpnamesvcs.com,168.183.36.43,UnitedHealth,4.228180e+11,United States,Health Care,5.0,168.183.36.0,22.0,10879,20111213.0,UHC,UNITED-215-ARIN,d39768e24c8472eb3b02486a17098b02_ARIN,ARIN
6,apple.com,a.ns.apple.com,17.253.200.1,Apple,4.086250e+11,United States,Information Technology,6.0,17.253.200.0,24.0,714,20250402.0,APPLE-ENGINEERING,APPLEC-1-Z-ARIN,c1e8d3f619ee04e379582a4ec734391d_ARIN,ARIN
7,apple.com,d.ns.apple.com,204.26.57.1,Apple,4.086250e+11,United States,Information Technology,6.0,204.26.57.0,24.0,42,20120302.0,WOODYNET-1,WOODYN-ARIN,cbe98e0e3a865d63238c7ae01041aab1_ARIN,ARIN
8,apple.com,c.ns.apple.com,204.19.119.1,Apple,4.086250e+11,United States,Information Technology,6.0,204.19.119.0,24.0,42,20120302.0,WOODYNET-1,WOODYN-ARIN,cbe98e0e3a865d63238c7ae01041aab1_ARIN,ARIN
9,berkshirehathaway.com,ns9.worldnic.com,162.159.26.110,Berkshire Hathaway,3.971460e+11,United States,Financials,7.0,162.159.26.0,24.0,13335,20170217.0,CLOUDFLARENET,CLOUD14-ARIN,28408e4f0567c2545afefd9cbce183bc_ARIN,ARIN


In [ ]:
def normalize_org_id(s):
    s = s.astype('string')  # tipo nativo string do pandas
    s = s.str.strip()
    s = s.str.upper()
    s = s.where(~s.isin(['NAN', 'NONE', 'NULL', '']), None)  # volta a None
    return s

merged_ns_as_org['org_id'] = normalize_org_id(merged_ns_as_org['org_id'])
as_org_to_info_dataset['org_id'] = normalize_org_id(as_org_to_info_dataset['org_id'])



# 3) Realizar o merge com o dataset as-to-org2info
merged_ns_as_org["org_id"] = merged_ns_as_org["org_id"].astype(str)

as_org_to_info_dataset["org_id"] = as_org_to_info_dataset["org_id"].astype(str)

#USE closest match LEFT JOIN

final_merged_dataset = pd.merge(
    merged_ns_as_org,
    as_org_to_info_dataset,
    on="org_id",
    how="left"
)

print(f"Number of unique domains with revenue Greater than 0: {final_merged_dataset[final_merged_dataset['revenue'] > 0]['domain'].nunique()}")
display(final_merged_dataset.head(10))

# Salvar o dataset final em um novo arquivo CSV
final_merged_dataset.to_csv(r'.\org_info_pipeline_multi_ns_enriched.csv', index=False)

Number of unique domains with revenue Greater than 0: 10374


,domain,NS,NS-A,company_name,revenue,country_name,sector,rank,prefix,prefix_length,as_number,changed_x,aut_name,org_id,opaque_id,source_x,changed_y,org_name,country,source_y
0,walmart.com,a5-65.akam.net,95.100.168.65,Walmart,6.931530e+11,United States,Consumer Staples,1.0,95.100.168.0,24.0,21342,NaN,AKAMAI-ASN2,ORG-AT1-RIPE,NaN,RIPE,NaN,Akamai International B.V.,NL,RIPE
1,walmart.com,pdnswm4.ultradns.org,156.154.67.4,Walmart,6.931530e+11,United States,Consumer Staples,1.0,156.154.67.0,24.0,12008,20221213.0,SECURITYSERVICES,SSL-1134-ARIN,d46edd86fd49dba22360500c6b8657bf_ARIN,ARIN,20250212.0,"Vercara, LLC",US,ARIN
2,walmart.com,a8-66.akam.net,2.16.40.66,Walmart,6.931530e+11,United States,Consumer Staples,1.0,2.16.40.0,24.0,21342,NaN,AKAMAI-ASN2,ORG-AT1-RIPE,NaN,RIPE,NaN,Akamai International B.V.,NL,RIPE
3,amazon.com,ns2.amzndns.net,156.154.69.10,Amazon,4.447496e+11,China,Energy,4.0,156.154.69.0,24.0,12008,20221213.0,SECURITYSERVICES,SSL-1134-ARIN,d46edd86fd49dba22360500c6b8657bf_ARIN,ARIN,20250212.0,"Vercara, LLC",US,ARIN
4,unitedhealthgroup.com,epla1.corpnamesvcs.com,149.111.128.204,UnitedHealth,4.228180e+11,United States,Health Care,5.0,149.111.128.0,19.0,10879,20111213.0,UHC,UNITED-215-ARIN,d39768e24c8472eb3b02486a17098b02_ARIN,ARIN,20111213.0,Unitedhealthcare,US,ARIN
5,unitedhealthgroup.com,epla2.corpnamesvcs.com,168.183.36.43,UnitedHealth,4.228180e+11,United States,Health Care,5.0,168.183.36.0,22.0,10879,20111213.0,UHC,UNITED-215-ARIN,d39768e24c8472eb3b02486a17098b02_ARIN,ARIN,20111213.0,Unitedhealthcare,US,ARIN
6,apple.com,a.ns.apple.com,17.253.200.1,Apple,4.086250e+11,United States,Information Technology,6.0,17.253.200.0,24.0,714,20250402.0,APPLE-ENGINEERING,APPLEC-1-Z-ARIN,c1e8d3f619ee04e379582a4ec734391d_ARIN,ARIN,20250422.0,Apple Inc.,US,ARIN
7,apple.com,d.ns.apple.com,204.26.57.1,Apple,4.086250e+11,United States,Information Technology,6.0,204.26.57.0,24.0,42,20120302.0,WOODYNET-1,WOODYN-ARIN,cbe98e0e3a865d63238c7ae01041aab1_ARIN,ARIN,20220428.0,"WoodyNet, Inc.",US,ARIN
8,apple.com,c.ns.apple.com,204.19.119.1,Apple,4.086250e+11,United States,Information Technology,6.0,204.19.119.0,24.0,42,20120302.0,WOODYNET-1,WOODYN-ARIN,cbe98e0e3a865d63238c7ae01041aab1_ARIN,ARIN,20220428.0,"WoodyNet, Inc.",US,ARIN
9,berkshirehathaway.com,ns9.worldnic.com,162.159.26.110,Berkshire Hathaway,3.971460e+11,United States,Financials,7.0,162.159.26.0,24.0,13335,20170217.0,CLOUDFLARENET,CLOUD14-ARIN,28408e4f0567c2545afefd9cbce183bc_ARIN,ARIN,20241125.0,"Cloudflare, Inc.",US,ARIN
